# Upload Data to TIDBRepo

## Module-Loading

In [108]:

from dbrepo.RestClient import RestClient
import urllib3
import pandas as pd


In [109]:

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


## Create Client
Create a username and password on https://tidbrepo.usm.my if you do not already have one. You should be able to create a container ID when you register.

In [110]:

client = RestClient(
    endpoint = "https://tidbrepo.usm.my",
    username = "yusriy",
    password = "xowzah-fatmi3-Nugxow",
    secure = False
)
    

### List the container ID

In [75]:
client.get_containers()

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_galera_11_3_2', running=None, hash=None)]

In [76]:

cont_id = client.get_containers()[0].id


## Create the Database

You can create the database programmatically, but an error will show up even though the database has been created.

In [ ]:

#db = client.create_database(
#    name = "env_forensics_data",
#    container_id = cont_id,
#    is_public = True
#)


List all databases. Confirm that the database has been created.

In [77]:

dbs = client.get_databases()
for d in dbs:
    print(d.name, d.id)
    

env_forensics_data 306d4236-97b2-4a17-a82d-957ac987afac
env_forensics_data be843444-e4b5-451a-ac15-1b8020d0e385
LST Analysis 66bc782d-964a-4f2c-b182-bf28d0cd669d
mkh_eddy_covariance f202bc5d-738c-41d7-b40d-a6fcd047a0ca
analysis_ch4_gosat_tropomi_heng_2026 bdb03710-4c1c-46b2-9b24-122d6bf794be
Sungai Dua, Universiti Sains Malaysia, Water Quality 7f048d8c-35d4-4deb-9252-d1a04937594b
Deep Learning Bias Correction in WRF adc8337c-718e-4653-aaaf-92d4bdf23c1c
emission_factor_malaysia 8cb5b734-7d57-4ca5-81a6-4d1de7acee88
MKH_water_temperature eff62402-14d8-4f8b-884b-ae6735875bd6


In [78]:

db_id = client.get_databases()[0].id
print(db_id)


306d4236-97b2-4a17-a82d-957ac987afac


### Load the data

In [70]:

df = pd.read_csv('class_honey_dat.csv')
print(df.head(10))


    Sample ID    Floral      Country Island  Year   Latitude   Longitude  \
0  100002 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
1  100003 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
2  100004 Avg    Manuka  New Zealand  North  2009 -39.054490  175.500869   
3  100005 Avg    Manuka  New Zealand  North  2009 -39.010753  175.421975   
4  100006 Avg    Kamahi  New Zealand  North  2009 -38.585933  175.454261   
5  100007 Avg    Kamahi  New Zealand  North  2009 -39.004214  175.483725   
6  100008 Avg    Kamahi  New Zealand  North  2009 -38.572419  175.441154   
7  100009 Avg    Manuka  New Zealand  North  2009 -35.385801  173.521300   
8  100010 Avg    Manuka  New Zealand  North  2009 -36.014152  173.551128   
9  100011 Avg  Honeydew  New Zealand  South  2009 -42.425167  172.332709   

         Li         B          Na  ...        Ga        Rb        Sr  \
0  0.043927  2.961700   25.253898  ...  0.017363  0.897764  0.127553   
1  0.014512  4.1399

#### Standardize column names

In [80]:

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("/","_", regex=False)
    .str.replace(" ","_", regex=False)
)
print(df.head(10))


    sample_id    floral      country island  year   latitude   longitude  \
0  100002 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
1  100003 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
2  100004 Avg    Manuka  New Zealand  North  2009 -39.054490  175.500869   
3  100005 Avg    Manuka  New Zealand  North  2009 -39.010753  175.421975   
4  100006 Avg    Kamahi  New Zealand  North  2009 -38.585933  175.454261   
5  100007 Avg    Kamahi  New Zealand  North  2009 -39.004214  175.483725   
6  100008 Avg    Kamahi  New Zealand  North  2009 -38.572419  175.441154   
7  100009 Avg    Manuka  New Zealand  North  2009 -35.385801  173.521300   
8  100010 Avg    Manuka  New Zealand  North  2009 -36.014152  173.551128   
9  100011 Avg  Honeydew  New Zealand  South  2009 -42.425167  172.332709   

         li         b          na  ...        ga        rb        sr  \
0  0.043927  2.961700   25.253898  ...  0.017363  0.897764  0.127553   
1  0.014512  4.1399

#### Set index (primary key)

In [81]:

df = df.set_index("sample_id")
df.index.name = "sample_id"


### Create Table without Data

In [31]:

table = client.create_table(
    database_id=db_id,
    name="honey_elemental_analysis",
    is_public=True,
    is_schema_public=True,
    dataframe=df,
    description="Elemental analysis of honey samples by ICP OES.",
    with_data=False
)


2026-04-10 10:53:09,362 root         WARNING default to 'text' for column sample_id and type <class 'numpy.dtype'>
2026-04-10 10:53:09,364 root         WARNING default to 'text' for column Floral and type <class 'numpy.dtype'>
2026-04-10 10:53:09,367 root         WARNING default to 'text' for column Country and type <class 'numpy.dtype'>
2026-04-10 10:53:09,368 root         WARNING default to 'text' for column Island and type <class 'numpy.dtype'>


### Check Table ID

In [82]:

tables = client.get_tables(db_id)

for t in tables:
    print(t.name, t.id)


honey_elemental_analysis 5fc7f10f-1627-4967-8c10-e8b05efc07d7


### Assign Table ID

In [83]:

table_id = next(t.id for t in tables if t.name == "honey_elemental_analysis")
print(table_id)


5fc7f10f-1627-4967-8c10-e8b05efc07d7


#### Insert Rows

##### Clean the Data First

In [85]:

import pandas as pd
import numpy as np

df_insert = df.reset_index()

# replace inf/-inf first
df_insert = df_insert.replace([np.inf, -np.inf], None)

# then replace NaN with None, using the same dataframe as the mask
df_insert = df_insert.where(pd.notnull(df_insert), None)

print(df_insert.head(10))


    sample_id    floral      country island  year   latitude   longitude  \
0  100002 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
1  100003 Avg    Manuka  New Zealand  North  2009 -39.445855  175.151239   
2  100004 Avg    Manuka  New Zealand  North  2009 -39.054490  175.500869   
3  100005 Avg    Manuka  New Zealand  North  2009 -39.010753  175.421975   
4  100006 Avg    Kamahi  New Zealand  North  2009 -38.585933  175.454261   
5  100007 Avg    Kamahi  New Zealand  North  2009 -39.004214  175.483725   
6  100008 Avg    Kamahi  New Zealand  North  2009 -38.572419  175.441154   
7  100009 Avg    Manuka  New Zealand  North  2009 -35.385801  173.521300   
8  100010 Avg    Manuka  New Zealand  North  2009 -36.014152  173.551128   
9  100011 Avg  Honeydew  New Zealand  South  2009 -42.425167  172.332709   

         li         b          na  ...        ga        rb        sr  \
0  0.043927  2.961700   25.253898  ...  0.017363  0.897764  0.127553   
1  0.014512  4.1399

#### Insert the Data Row-by-Row

In [68]:

failed_rows = []

for i, row in df_insert.iterrows():
    try:
        data = {
            k: (v.item() if hasattr(v, "item") else v)
            for k, v in row.to_dict().items()
        }

        client.create_table_data(
            database_id=db_id,
            table_id=table_id,
            data=data
        )

        if i % 10 == 0:
            print(f"Inserted row {i}")

    except Exception as e:
        failed_rows.append((i, str(e)))
        print(f"Failed row {i}: {e}")

print("Done")
print("Failed rows:", len(failed_rows))


Inserted row 0
Inserted row 10
Inserted row 20
Inserted row 30
Inserted row 40
Done
Failed rows: 0


## Publish the Identifier (PID)

In [118]:
import requests

payload = {
    "database_id": db_id,
    "type": "database",
    "titles": [
        {
            "title": "Environmental Forensics Data: Honey Elemental Analysis",
            "language": "en",
            "type": None
        }
    ],
    "descriptions": [
        {
            "description": "Elemental analysis of honey samples using ICP OES. Data provided by Dr. Syahidah Akmal Muhammad.",
            "language": "en",
            "type": "Abstract"
        },
        {
            "description": "Measured by ICP OES. Samples of honey.",
            "language": "en",
            "type": "Methods"
        },
        {
            "description": "Multiple columns of elemental concentrations of honey samples with locations.",
            "language": "en",
            "type": "Other"
        }
    ],
    "funders": [],
    "publisher": "School of Industrial Technology, Universiti Sains Malaysia",
    "licenses": [
        {
            "identifier": "CC0-1.0",
            "uri": "https://creativecommons.org/publicdomain/zero/1.0/legalcode",
            "description": "CC0 waives copyright interest in a work you've created and dedicates it to the world-wide public domain. Use CC0 to opt out of copyright entirely and ensure your work has the widest reach."
        }
    ],
    "creators": [
        {
            "firstname": "Yusri",
            "lastname": "Yusup",
            "creator_name": "Yusup, Yusri",
            "name_type": "Personal",
            "name_identifier": "https://orcid.org/0000-0001-6703-2208",
            "affiliation": "Universiti Sains Malaysia",
            "affiliation_identifier": None
        }
    ],
    "publication_day": 10,
    "publication_month": 4,
    "publication_year": 2026,
    "related_identifiers": []
}

response = requests.post(
    f"{client.endpoint}/api/v1/identifier",
    auth=(client.username, client.password),
    verify=client.secure,
    headers={"Accept": "application/json", "Content-Type": "application/json"},
    json=payload
)

print(response.status_code)
print(response.text)

201
{"id":"e9d937b4-4ec1-49c7-889a-b6e0fc4d62e7","links":{"self":"/api/v1/identifier/e9d937b4-4ec1-49c7-889a-b6e0fc4d62e7","data":null,"self_html":"/pid/e9d937b4-4ec1-49c7-889a-b6e0fc4d62e7","dashboard_html":"/d/afil3fhr7mwaoe"},"type":"database","titles":[{"id":"882087de-a123-4ec8-a783-9d5355791c74","title":"Environmental Forensics Data: Honey Elemental Analysis","language":"en","type":null}],"descriptions":[{"id":"10ddfc76-9498-4a43-bb7e-a7e73765de93","description":"Elemental analysis of honey samples using ICP OES. Data provided by Dr. Syahidah Akmal Muhammad.","language":"en","type":"Abstract"},{"id":"21096be0-b8f1-4b5d-bb65-d01695fb42fa","description":"Measured by ICP OES. Samples of honey.","language":"en","type":"Methods"},{"id":"95cae1e6-d82a-4524-9bd2-ff7adafc6ff6","description":"Multiple columns of elemental concentrations of honey samples with locations.","language":"en","type":"Other"}],"funders":[],"query":null,"execution":null,"doi":null,"publisher":"School of Industrial 

### Publish It

Publish it on the website GUI.

## Add Semantics to the Table

In [119]:
table_meta = client.get_table(db_id,table_id)

In [121]:
col_map = {col.internal_name: col.id for col in table_meta.columns}
print(col_map)

{'sample_id': 'f6f0efdc-ba2c-4a94-8493-18d82869d077', 'floral': '2dddbfe5-ee1e-49e0-9926-8329837f44bb', 'country': '37f49bdf-7fa8-47a9-a084-ebe574047ef9', 'island': '77d76298-404d-4ec9-8201-d6d5e6be0b91', 'year': 'ce1c58f6-65f0-4ba7-9513-e6a1665a3a1a', 'latitude': 'ce7de7f3-c957-4f42-a575-b83593bed48e', 'longitude': '748e8888-e4f7-4932-9110-46bbedb968ac', 'li': '8db547d7-8bbb-4619-bec7-a3092a701502', 'b': '53517ee7-3593-47c3-9e42-b61a0c31779d', 'na': '1b8fa19b-cbb1-462d-b0fd-a4347ceb634d', 'mg': 'accfd801-b399-4e89-90db-5b88bde2cfd1', 'al': 'c8ea2198-024d-4c52-8bb9-7e170f858923', 'k': 'b5b8568c-e2e8-4c5c-82ef-6ed83e90d10f', 'ca': '72fd94a4-e458-45a7-b735-1bcc2bcab59d', 'v': '8680d960-7765-4674-92e2-2a1ab1ce8e9d', 'cr': 'c95d0db5-b16f-4e89-8518-916d80cc4ab8', 'mn': '588e43d7-c9f6-428d-8db8-1530506d1145', 'fe': 'bef55cf3-2b43-4d98-bf8c-7e1403dbd74d', 'co': '23949315-4f97-4ac7-b53a-dad4ebc23a66', 'ni': '1a612bf7-87b7-43a0-bafa-d710f5d37154', 'cu': 'c5fe9d69-6ea0-4b68-bd11-d8cd8b9f1b27', '

In [127]:
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["longitude"],
    concept_uri="http://dbpedia.org/ontology/longitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree",
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

Column(id='748e8888-e4f7-4932-9110-46bbedb968ac', name='Longitude', database_id='306d4236-97b2-4a17-a82d-957ac987afac', table_id='5fc7f10f-1627-4967-8c10-e8b05efc07d7', ord=6, internal_name='longitude', is_null_allowed=False, type=<ColumnType.DECIMAL: 'decimal'>, alias=None, description=None, size=40, d=20, mean=None, median=None, concept=ConceptBrief(id='7cf73a98-2f77-468b-9a54-41b94694ceb0', uri='http://dbpedia.org/ontology/longitude', name=None, description=None), unit=UnitBrief(id='7fe2598d-6bdb-4727-871e-6710ec833d25', uri='http://www.ontology-of-units-of-measure.org/resource/om-2/degree', name=None, description=None), enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None)

## Identifiers / Categorical

In [130]:

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["sample_id"],
    concept_uri="http://dbpedia.org/ontology/identifier",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["floral"],
    concept_uri="http://dbpedia.org/ontology/Plant",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["country"],
    concept_uri="http://dbpedia.org/ontology/Country",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["island"],
    concept_uri="http://dbpedia.org/ontology/Island",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["year"],
    concept_uri="http://dbpedia.org/ontology/date",
    unit_uri=None
)

# --- spatial ---
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["longitude"],
    concept_uri="http://dbpedia.org/ontology/longitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

# --- elemental concentrations (ppm) ---
ppm = "http://www.ontology-of-units-of-measure.org/resource/om-2/partsPerMillion"

client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["li"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["b"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["na"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["mg"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["al"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["k"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ca"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["v"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cr"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["mn"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["fe"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["co"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ni"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cu"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["zn"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ga"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["rb"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["sr"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ag"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cs"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ba"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["la"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ce"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["pb"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)

# --- ratio ---
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["rb_sr"],
    concept_uri="http://dbpedia.org/ontology/ratio",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/one"
)

Column(id='22824ef3-e55c-47fb-8d35-81eb0873d955', name='Rb/Sr', database_id='306d4236-97b2-4a17-a82d-957ac987afac', table_id='5fc7f10f-1627-4967-8c10-e8b05efc07d7', ord=31, internal_name='rb_sr', is_null_allowed=False, type=<ColumnType.DECIMAL: 'decimal'>, alias=None, description=None, size=40, d=20, mean=None, median=None, concept=ConceptBrief(id='e9bf8633-a5ef-4a45-ad46-f8efdf714b6d', uri='http://dbpedia.org/ontology/ratio', name=None, description=None), unit=UnitBrief(id='7ae93642-7693-4cce-99d7-34e2a2b16e92', uri='http://www.ontology-of-units-of-measure.org/resource/om-2/one', name=None, description=None), enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None)